# FOMC Multimodal Sentiment — Analysis Report

This notebook runs the distributed sentiment pipeline over the events in `data/events.json`,
collects per-channel scores (NLP, audio, vision) and the fused signal from the gateway,
and compares each against the contemporaneous S&P 500 percentage move.

> **Honest-scope caveat:** The sample is small (3–5 events). Any correlations observed
> are **illustrative only and not statistically significant**. A minimum of ~30 events
> would be needed for even preliminary inference. Treat outputs as a proof-of-concept
> demonstration of the multimodal pipeline, not as a research finding.

In [ ]:
import json
import httpx
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

events = json.loads(Path("data/events.json").read_text())
rows = []
for ev in events:
    res = httpx.post("http://localhost:8000/analyze", json={"event_id": ev["id"]}, timeout=600).json()
    by = {c["channel"]: c["score"] for c in res["channels"]}
    signal = json.loads(Path(f"data/processed/{ev['id']}/market_signal.json").read_text())["signal_pct"]
    rows.append({
        "event": ev["id"],
        "nlp": by.get("nlp"),
        "audio": by.get("audio"),
        "vision": by.get("vision"),
        "fusion": res["combined_score"],
        "market": signal
    })
df = pd.DataFrame(rows)
df

In [ ]:
corr = {ch: df[ch].corr(df["market"]) for ch in ["nlp", "audio", "vision", "fusion"]}
corr_df = pd.DataFrame.from_dict(corr, orient="index", columns=["corr_with_market"])
corr_df

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, ch in zip(axes, ["nlp", "audio", "vision", "fusion"]):
    ax.scatter(df[ch], df["market"])
    ax.set_title(f"{ch} vs market")
    ax.set_xlabel("sentiment")
    ax.set_ylabel("S&P % move")
plt.tight_layout()
plt.savefig("report_correlations.png", dpi=120)
plt.show()

## Interpretation

The scatter plots and correlation table above show the relationship between each sentiment
channel and the S&P 500 move on/after the FOMC press conference day.

- **NLP (FinBERT):** Captures textual tone of the transcript. A positive correlation would
  suggest markets respond positively to dovish language.
- **Audio (Wav2Vec2):** Captures emotional valence in Powell's voice. Prosodic cues may
  carry information beyond the literal words.
- **Vision (DeepFace):** Captures facial affect. This channel is best-effort — it may fail
  (`ok: false`) for events where face detection is unreliable, and is excluded from fusion.
- **Fusion:** Weighted combination (NLP 50%, audio 30%, vision 20%, renormalized if a
  channel fails). Expected to be more stable than any individual channel.

> **Caveat (repeated):** With only 3–5 data points, no correlation is statistically
> meaningful. These results are a proof-of-concept demonstration. To draw any conclusion,
> extend `data/events.json` to cover 30+ FOMC events and re-run `data/prepare_data.py`.